## Previsão de Rotatividade de Clientes
### Projeto Didático — Data Science & Machine Learning Clássico

---

**Objetivo:** construir um pipeline completo de Data Science para prever quais clientes de uma empresa de telecomunicações têm maior probabilidade de cancelar o serviço (*churn*), permitindo ações de retenção proativas.

**Dataset:** [Telco Customer Churn](https://www.kaggle.com/blastchar/telco-customer-churn) (IBM Sample Dataset) — 7.043 clientes, 21 atributos.

**O que este notebook cobre:**
1. Contextualização do problema de negócio
2. Análise Exploratória de Dados (EDA)
3. Limpeza e pré-processamento
4. Engenharia de features
5. Tratamento de desbalanceamento de classes
6. Modelagem: Regressão Logística → Random Forest → XGBoost
7. Avaliação com métricas apropriadas (não só acurácia!)
8. Interpretação de resultados e importância das variáveis
9. Conclusões e próximos passos

## 1. Contexto do Problema de Negócio

**Churn** (ou taxa de cancelamento) é uma das métricas mais críticas para empresas baseadas em assinatura (telecom, streaming, SaaS, etc.). Adquirir um novo cliente custa, em média, de 5 a 25 vezes mais do que reter um cliente existente.

**Pergunta de negócio:** *Dado o perfil e o histórico de um cliente, qual a probabilidade de ele cancelar o serviço no próximo período?*

**Por que isso importa:**
- Permite à empresa priorizar ações de retenção (descontos, contato proativo, upgrade de plano) para clientes de alto risco.
- Evita desperdiçar recursos de retenção em clientes que não iriam cancelar de qualquer forma.

**Tipo de problema:** Classificação binária supervisionada (`Churn` = Yes/No).

**Desafio central deste dataset:** as classes são desbalanceadas (~73% não-churn vs ~27% churn), então acurácia sozinha é uma métrica enganosa — um modelo "preguiçoso" que sempre prevê "não vai cancelar" já acerta 73% das vezes sem aprender nada útil.


## 2. Setup do Ambiente

Importar as bibliotecas necessárias. Se estiver rodando localmente pela primeira vez, instale as dependências:

```bash
pip install pandas numpy matplotlib seaborn scikit-learn xgboost imbalanced-learn shap jinja2
```


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Configurações visuais
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Reprodutibilidade
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Ambiente configurado com sucesso.')

## 3. Carregamento dos Dados

O dataset "Telco Customer Churn" é público (IBM Sample Data Sets). Neste projeto vamos carregá-lo diretamente.


In [ ]:
# Caminho do arquivo local (ajuste conforme necessário)
CAMINHO_CSV = 'Telco-Customer-Churn.csv'

df = pd.read_csv(CAMINHO_CSV)

print(f'Shape do dataset: {df.shape[0]} linhas x {df.shape[1]} colunas')
df.head()

In [ ]:
df.info()

In [ ]:
# Estatísticas descritivas das variáveis numéricas
df.describe()

## 4. Limpeza dos Dados

Alguns pontos de atenção conhecidos neste dataset (bugs "clássicos" que todo cientista de dados deve saber identificar):

1. **`TotalCharges`** vem como `object` (string), não `float` — porque existem 11 registros com espaço em branco `" "` no lugar de um número (clientes com `tenure = 0`, ou seja, cadastrados mas ainda sem cobrança).
2. **`customerID`** é um identificador único, sem valor preditivo — deve ser removido antes da modelagem (mas guardado para referência).
3. Várias colunas categóricas usam `"No internet service"` / `"No phone service"` como uma categoria distinta de `"No"` — vamos decidir conscientemente se simplificamos isso.


In [ ]:
# 1. Corrigindo TotalCharges
print('Valores não numéricos em TotalCharges:', (df['TotalCharges'].str.strip() == '').sum())

# Converte para numérico, forçando erros para NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(f'Valores nulos após conversão: {df["TotalCharges"].isnull().sum()}')
print(df[df['TotalCharges'].isnull()][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']])

In [ ]:
# Confirma a hipótese: todos os NaN são clientes com tenure == 0 (acabaram de entrar)
assert (df.loc[df['TotalCharges'].isnull(), 'tenure'] == 0).all(), 'Hipótese não confirmada!'

# Para esses clientes, faz sentido que TotalCharges seja 0 (ainda não foram cobrados)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print('Nulos restantes em TotalCharges:', df['TotalCharges'].isnull().sum())

In [ ]:
# 2. Guardamos customerID à parte e removemos do dataframe de trabalho
customer_ids = df['customerID'].copy()
df = df.drop(columns=['customerID'])

# 3. Verificando duplicatas
print('Linhas duplicadas:', df.duplicated().sum())

# 4. Checagem final de nulos
df.isnull().sum()[df.isnull().sum() > 0]

## 5. Análise Exploratória de Dados (EDA)

### 5.1 Distribuição da variável-alvo (Churn)


In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(churn_counts.index, churn_counts.values, color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribuição Absoluta de Churn')
axes[0].set_ylabel('Número de Clientes')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

axes[1].pie(churn_pct.values, labels=churn_pct.index, autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Distribuição Percentual de Churn')

plt.tight_layout()
plt.show()

print(f'\nDesbalanceamento de classes: {churn_pct["No"]:.1f}% não-churn vs {churn_pct["Yes"]:.1f}% churn')
print('=> Isso confirma que acurácia sozinha NÃO é uma métrica confiável para este problema.')

### 5.2 Churn por variáveis categóricas relevantes

Vamos investigar como o churn se relaciona com tipo de contrato, método de pagamento e serviço de internet — hipóteses de negócio comuns.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

categorical_vars = ['Contract', 'PaymentMethod', 'InternetService', 'PaperlessBilling']

for ax, col in zip(axes.flatten(), categorical_vars):
    churn_rate = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).sort_values(ascending=False)
    bars = ax.bar(churn_rate.index, churn_rate.values, color='#3498db')
    ax.set_title(f'Taxa de Churn por {col}')
    ax.set_ylabel('Taxa de Churn (%)')
    ax.tick_params(axis='x', rotation=30)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.5, f'{h:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

**Insights esperados (devemos validar ao rodar):**
- Contratos `Month-to-month` tendem a ter churn muito mais alto que contratos anuais/bianuais (faz sentido: menor barreira de saída).
- `Electronic check` como método de pagamento costuma ter maior churn — pode indicar um segmento de clientes menos fidelizado.
- Fiber optic tende a ter mais churn que DSL — possivelmente por preço mais alto ou problemas de qualidade percebidos.


In [ ]:
# Distribuição de variáveis numéricas por churn
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

numeric_vars = ['tenure', 'MonthlyCharges', 'TotalCharges']

for ax, col in zip(axes, numeric_vars):
    sns.kdeplot(data=df, x=col, hue='Churn', fill=True, alpha=0.4, ax=ax,
                palette={'No': '#2ecc71', 'Yes': '#e74c3c'})
    ax.set_title(f'Distribuição de {col} por Churn')

plt.tight_layout()
plt.show()

**Insight chave:** clientes com **baixo `tenure`** (poucos meses de casa) têm risco de churn muito maior. Isso sugere que o período inicial do cliente é crítico para retenção — um achado clássico e acionável neste tipo de negócio.


In [ ]:
# Matriz de correlação das variáveis numéricas
df_corr = df.copy()
df_corr['Churn_binary'] = (df_corr['Churn'] == 'Yes').astype(int)

corr_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn_binary']
corr_matrix = df_corr[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlação (variáveis numéricas + Churn)')
plt.tight_layout()
plt.show()

## 6. Engenharia de Features

Vamos criar variáveis derivadas que capturam sinais de negócio úteis, além de preparar o encoding para os modelos.

**Features novas propostas:**
- `tenure_group`: agrupamento do tempo de casa em faixas (0-12, 13-24, 25-48, 49+ meses) — modelos de árvore lidam bem com isso, mas também ajuda na interpretação.
- `num_services`: quantidade total de serviços adicionais contratados (Segurança, Backup, Streaming, etc.) — proxy de "engajamento" do cliente.
- `avg_charge_per_tenure`: `TotalCharges / (tenure + 1)` — ajuda a detectar inconsistências e captura o "ticket médio real" do cliente.


In [ ]:
df_feat = df.copy()

# --- Feature 1: faixa de tenure ---
def tenure_group(t):
    if t <= 12:
        return '0-12'
    elif t <= 24:
        return '13-24'
    elif t <= 48:
        return '25-48'
    else:
        return '49+'

df_feat['tenure_group'] = df_feat['tenure'].apply(tenure_group)

# --- Feature 2: número de serviços adicionais contratados ---
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                 'TechSupport', 'StreamingTV', 'StreamingMovies']

df_feat['num_services'] = (df_feat[service_cols] == 'Yes').sum(axis=1)

# --- Feature 3: ticket médio real por mês de casa ---
df_feat['avg_charge_per_tenure'] = df_feat['TotalCharges'] / (df_feat['tenure'] + 1)

print('Novas features criadas:', ['tenure_group', 'num_services', 'avg_charge_per_tenure'])
df_feat[['tenure', 'tenure_group', 'num_services', 'avg_charge_per_tenure']].head()

In [ ]:
# Validação rápida: num_services realmente separa as classes?
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df_feat, x='num_services', y='MonthlyCharges', hue='Churn', ax=ax,
            palette={'No': '#2ecc71', 'Yes': '#e74c3c'})
ax.set_title('MonthlyCharges por Número de Serviços e Churn')
plt.tight_layout()
plt.show()

### 6.1 Encoding das variáveis categóricas

- **Variáveis binárias** (`Yes`/`No`, `Male`/`Female`) → codificação 0/1 direta.
- **Variáveis categóricas nominais** (mais de 2 categorias, sem ordem) → One-Hot Encoding.
- **`tenure_group`** (ordinal) → poderia ser ordinal, mas para simplicidade e para não impor uma relação linear artificial, também usaremos one-hot.


In [ ]:
# Separando o alvo
y = (df_feat['Churn'] == 'Yes').astype(int)
X = df_feat.drop(columns=['Churn'])

# Identificando tipos de colunas
# Usamos select_dtypes com ['object', 'string', 'category'] em vez de comparar
# diretamente com 'object', pois versoes mais novas do pandas podem usar o
# dtype nativo 'string' para colunas de texto (dependendo da configuracao/versao).
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
text_cols = X.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
multi_cat_cols = [c for c in text_cols if c not in binary_cols]

print('Colunas binárias:', binary_cols)
print('\nColunas categóricas multi-classe:', multi_cat_cols)

In [ ]:
# Encoding binário simples
binary_map = {
    'gender': {'Male': 1, 'Female': 0},
    'Partner': {'Yes': 1, 'No': 0},
    'Dependents': {'Yes': 1, 'No': 0},
    'PhoneService': {'Yes': 1, 'No': 0},
    'PaperlessBilling': {'Yes': 1, 'No': 0},
}

for col, mapping in binary_map.items():
    X[col] = X[col].map(mapping)

# One-Hot Encoding para as demais categóricas
X = pd.get_dummies(X, columns=multi_cat_cols, drop_first=True)

print(f'Shape final de X após encoding: {X.shape}')
X.head()

## 7. Divisão Treino/Teste

Usamos `stratify=y` para garantir que a proporção de churn seja preservada em ambos os conjuntos — essencial em problemas desbalanceados.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Treino: {X_train.shape[0]} amostras | Churn rate: {y_train.mean():.1%}')
print(f'Teste:  {X_test.shape[0]} amostras | Churn rate: {y_test.mean():.1%}')

> **Devemos te cuidado com vazamento de dados (data leakage):** todo pré-processamento que "aprende" algo dos dados (normalização, imputação por média, seleção de features) deve ser ajustado (`fit`) **apenas no treino** e depois aplicado (`transform`) no teste. Vamos seguir essa disciplina a partir daqui usando `Pipeline` do scikit-learn.


## 8. Modelo Baseline: Regressão Logística

Começamos com o modelo mais simples e interpretável possível. Ele serve como nossa **linha de base** — qualquer modelo mais complexo só se justifica se superar este baseline de forma consistente.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                              roc_curve, precision_recall_curve, f1_score)

pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'))
])

pipe_lr.fit(X_train, y_train)

y_pred_lr = pipe_lr.predict(X_test)
y_proba_lr = pipe_lr.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_lr, target_names=['No Churn', 'Churn']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba_lr):.4f}')

**Por que `class_weight='balanced'`?** Isso instrui o modelo a penalizar mais os erros na classe minoritária (churn), compensando o desbalanceamento sem precisar reamostrar os dados artificialmente. É uma alternativa mais simples ao SMOTE, que usaremos mais adiante para comparação.


## 9. Random Forest

Modelo de ensemble baseado em árvores, capaz de capturar relações não-lineares e interações entre variáveis sem exigir normalização dos dados.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf, target_names=['No Churn', 'Churn']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba_rf):.4f}')

> Note os hiperparâmetros `max_depth=8` e `min_samples_leaf=20`: são escolhas deliberadas para **limitar overfitting**. Com ~5.600 amostras de treino, uma Random Forest sem restrições tende a decorar o treino. Mais adiante, no Projeto 3 (ou em uma extensão deste), usaríamos `GridSearchCV`/Optuna para tunar isso de forma sistemática.


## 10. XGBoost

Gradient Boosting costuma superar Random Forest em dados tabulares por construir árvores sequencialmente, cada uma corrigindo os erros da anterior.


In [ ]:
from xgboost import XGBClassifier

# Calculando o peso da classe positiva para lidar com desbalanceamento
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight calculado: {scale_pos_weight:.2f}')

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_xgb, target_names=['No Churn', 'Churn']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba_xgb):.4f}')

## 11. Comparando Estratégias de Balanceamento: `class_weight` vs SMOTE

Até aqui usamos `class_weight='balanced'`/`scale_pos_weight` (ponderação de custo). Agora vamos comparar com **SMOTE** (Synthetic Minority Oversampling Technique), que gera exemplos sintéticos da classe minoritária.

> **SMOTE** deve ser aplicado **somente no conjunto de treino**, nunca no teste — senão estaríamos "vazando" informação sintética derivada dos próprios dados de teste.


In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

smote_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('clf', XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
    ))
])

smote_pipe.fit(X_train, y_train)

y_pred_smote = smote_pipe.predict(X_test)
y_proba_smote = smote_pipe.predict_proba(X_test)[:, 1]

print('Distribuição original do treino:', dict(y_train.value_counts()))
print()
print(classification_report(y_test, y_pred_smote, target_names=['No Churn', 'Churn']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba_smote):.4f}')

## 12. Validação Cruzada Estratificada

Um único split treino/teste pode ser sensível a "sorte" na divisão. Vamos usar `StratifiedKFold` para obter uma estimativa mais robusta do ROC-AUC de cada modelo.


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

modelos_cv = {
    'Regressão Logística': pipe_lr,
    'Random Forest': rf,
    'XGBoost': xgb,
}

resultados_cv = {}
for nome, modelo in modelos_cv.items():
    scores = cross_val_score(modelo, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    resultados_cv[nome] = scores
    print(f'{nome:22s} | ROC-AUC médio: {scores.mean():.4f} (+/- {scores.std():.4f})')

In [ ]:
plt.figure(figsize=(9, 5))
try:
    # Matplotlib >= 3.9 renomeou o parametro 'labels' para 'tick_labels'
    plt.boxplot(resultados_cv.values(), tick_labels=list(resultados_cv.keys()))
except TypeError:
    # Compatibilidade com versoes mais antigas do matplotlib
    plt.boxplot(resultados_cv.values(), labels=list(resultados_cv.keys()))
plt.title('ROC-AUC por Modelo (5-Fold Cross-Validation)')
plt.ylabel('ROC-AUC')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Comparação Final entre Modelos

Vamos consolidar todas as métricas relevantes em uma única tabela e visualizar as curvas ROC lado a lado.

**Por que múltiplas métricas?**
- **Accuracy**: enganosa em dados desbalanceados (um modelo "bobo" já acerta ~73%).
- **Precision**: dos clientes que o modelo previu como "vão cancelar", quantos realmente cancelaram? Importante se ações de retenção têm custo (ex: desconto).
- **Recall**: dos clientes que realmente cancelaram, quantos o modelo conseguiu identificar? Importante se o custo de "perder" um cliente é alto.
- **F1-score**: equilíbrio entre precision e recall.
- **ROC-AUC**: mede a capacidade do modelo de ranquear corretamente clientes de risco, independente do limiar de decisão escolhido.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

resultados = {
    'Regressão Logística': (y_pred_lr, y_proba_lr),
    'Random Forest': (y_pred_rf, y_proba_rf),
    'XGBoost': (y_pred_xgb, y_proba_xgb),
    'XGBoost + SMOTE': (y_pred_smote, y_proba_smote),
}

tabela_metricas = []
for nome, (y_pred, y_proba) in resultados.items():
    tabela_metricas.append({
        'Modelo': nome,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba),
    })

df_metricas = pd.DataFrame(tabela_metricas).set_index('Modelo').round(4)
df_metricas.style.background_gradient(cmap='Greens', axis=0)

In [ ]:
# Curvas ROC comparativas
plt.figure(figsize=(9, 7))

for nome, (y_pred, y_proba) in resultados.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{nome} (AUC = {auc:.3f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Classificador Aleatório (AUC = 0.5)', alpha=0.6)
plt.xlabel('Taxa de Falsos Positivos (FPR)')
plt.ylabel('Taxa de Verdadeiros Positivos (TPR / Recall)')
plt.title('Curvas ROC Comparativas entre Modelos')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Matrizes de confusão lado a lado
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

for ax, (nome, (y_pred, _)) in zip(axes, resultados.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Previsto: No', 'Previsto: Yes'],
                yticklabels=['Real: No', 'Real: Yes'], cbar=False)
    ax.set_title(nome, fontsize=10)

plt.tight_layout()
plt.show()

## 14. Importância das Variáveis (Feature Importance)

Além de prever, queremos **entender** quais fatores mais influenciam o churn — informação diretamente acionável para a área de negócio.


In [ ]:
importancias = pd.DataFrame({
    'feature': X.columns,
    'importance': xgb.feature_importances_
}).sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(10, 7))
sns.barplot(data=importancias, x='importance', y='feature', palette='viridis')
plt.title('Top 15 Features Mais Importantes (XGBoost)')
plt.xlabel('Importância')
plt.tight_layout()
plt.show()

> **Indo além:** para uma interpretação mais rigorosa (que considera direção e magnitude do efeito de cada feature em cada previsão individual), o próximo passo natural é usar **SHAP values**:
> ```python
> import shap
> explainer = shap.TreeExplainer(xgb)
> shap_values = explainer.shap_values(X_test)
> shap.summary_plot(shap_values, X_test)
> ```


## 15. Conclusões e Próximos Passos

### Resumo dos achados
- O dataset apresenta desbalanceamento de classes (~27% churn), o que exige métricas além da acurácia.
- Variáveis como **tipo de contrato**, **tempo de casa (tenure)** e **método de pagamento** são fortes preditores de churn.
- Modelos de ensemble (Random Forest, XGBoost) superam a Regressão Logística, mas a diferença nem sempre é enorme — vale sempre comparar com um baseline simples antes de justificar a complexidade extra.
- Balanceamento via `class_weight`/`scale_pos_weight` teve desempenho comparável ao SMOTE neste caso — vale testar ambos em cada novo problema, já que o resultado pode variar bastante conforme o dataset.

### Possíveis extensões futuras:
1. **Tuning de hiperparâmetros** com `GridSearchCV`, `RandomizedSearchCV` ou Optuna.
2. **SHAP values** para interpretabilidade individual das previsões.
3. **Ajuste do limiar de decisão** (threshold tuning) com base no custo de negócio real (ex: custo de uma ação de retenção vs. custo de perder um cliente).
4. **Deploy** simples via Streamlit, permitindo simular o risco de churn de um cliente hipotético interativamente.
5. Testar **LightGBM** ou **CatBoost** como alternativas ao XGBoost.



### Contato: 
Marcelo Saorim - marcelo.saoriml@gmail.com

### LinkedIn: 
[marcelo-rocha-saorim](https://www.linkedin.com/in/marcelo-rocha-saorim/)

### GitHub: 
[@saorim10](https://github.com/saorim10)